# Trabajo Práctico Nro. 2A. Rev. 3
### Ing. Javier Ouret - UCA - Facultad de Ingeniería
## Monitoreo de dispositivos y software en redes cliente-servidor utilizando SNMP

### Realizar ejercicios 1, 2, 3 y 4

### 1 Activar y configurar SNMP en Ubuntu (snmpd)

**snmpd es un agente SNMP**: software que se ejecuta en un dispositivo gestionado (como un servidor, switch, router, etc.) y que expone información del sistema mediante el protocolo SNMP. Responde a las peticiones del gestor SNMP (Zabbix, Nagios, LibreNMS) enviando datos o aceptando comandos.  

### 2 Instalar un agente SNMP gratuito con interfaz web en Linux

##### Herramientas útiles para trabajar con MIBs
- snmptranslate: para ver los OID en texto y número.
- snmpwalk/snmpget: para consultar datos reales con SNMP.
- Cacti/Zabbix/Nagios/PRTG: para integrarlas a sistemas de monitoreo.
- MIB Browsers: iReasoning o SNMP MIB Browser de ManageEngine.

##### Hacemos una prueba con Cacti, que es una agente SNMP (CPU, memoria, interfaces) focalizado en gráficas de monitoreo.

- Cacti
  - 
  - Más liviano que LibreNMS.
- Instalación en Ubuntu:

- Durante la instalación de cacti, el instalador permite configurar la base de datos con dbconfig-common. 
- Seleccionar "Sí" e indicar una contraseña para el usuario de MySQL cacti.
- Si no aparece, puedes hacerlo manualmente con:
sudo mysql -u root -p

Dentro de mysql hacer lo siguiente:

```sql
CREATE DATABASE cacti;
GRANT ALL PRIVILEGES ON cacti.* TO 'cactiuser'@'localhost' IDENTIFIED BY 'clave';
FLUSH PRIVILEGES;
EXIT;
```

Importar el esquema

sudo mysql -u root -p cacti < /usr/share/doc/cacti/cacti.sql

Editar el archivo de configuración para incluir usuario y clave

Editar el cron de linux para que actualice los gráficos y que tiene la siguiente línea habilitada (sin #):

Editar nuevamente la configuración de snmpd para inidcar rack simulado y contacto. Luego reiniciar snmpd.

Probar si anda snmpd

In [14]:
!snmpwalk -v2c -c public localhost

Bad operator (INTEGER): At line 73 in /usr/share/snmp/mibs/ietf/SNMPv2-PDU
Bad operator (FROM): At line 9 in /usr/share/snmp/mibs/MY-SENSOR-MIB.txt
SNMPv2-SMI::mib-2 = No more variables left in this MIB View (It is past the end of the MIB tree)


#### Acceder a: http://localhost/cacti
Tener snmpd activo en el servidor Ubuntu a monitorear.

Agregar dispositivo SNMP en localhost

- En el panel de Cacti:
  - Ve a Devices → Add
  - Nombre: localhostHostname: 127.0.0.1
  - Template: Generic SNMP Device
  - SNMP Version: Version 2
  - Community: public
  - Guardar y luego en la opción "Create Graphs for this Host", seleccionar los gráficos a generar.

- Ver Gráficos
  - Una vez creados los gráficos, esperar unos minutos para que Cacti recoja datos.
  - Ir a Graphs
  - Buscar el host
  - Ver el gráfico generado

### 3 Instalar un browser SNMP en Windows
- Nota: En Windows primero hay que instalar SNMP y activarlo con PowerShell como administrador

#### Descargar MIB Browser para Windows y para Linux

- MIB Simulada (SimpleMIB): Esta clase almacena los valores de los OIDs simulados. En este caso, hemos definido un solo OID, que es el estado de un dispositivo (con un valor de 1, indicando "ON").
- Operaciones GET y SET: La función handle_get_request() simula una solicitud GET, y la función handle_set_request() simula una solicitud SET.
- Servidor SNMP: El servidor SNMP virtual se configura para escuchar en el puerto 161 (el puerto estándar para SNMP). En este ejemplo, está diseñado para simular respuestas GET y SET.

https://www.ireasoning.com/mibbrowser.shtml

Configuración básica:

Mofidicar la configuración del MIB Browser   
En address agregar "localhost"   

![image](https://github.com/jaouret/PDI_2024-TP1-TP2-TP3-TP4/assets/111520053/ad5e4c2c-04a2-4dc2-830c-57262cdfb967)   
En advanced en los campos read community y write community agregar COMUNIDAD_PDI. Y la version 2   
![image](https://github.com/jaouret/PDI_2024-TP1-TP2-TP3-TP4/assets/111520053/36b36bb8-9f46-438b-87fd-944f64ab80c7)   

### 4 Simular un Cliente SNMP con una MIB personalizada

##### Objetivo:
- Completar la instalación de python para poder usar SNMP como cliente de un agente SNMP (servidor).
- Utilizar la librería pysnmp, que permite manejar la gestión de SNMP.
- Consultar con un cliene SNMP al equipo con snmpd usando python y también desde gestor snmpd gratuito web (Cacti).
- Crear una MIB personalizada y realizar las mismas pruebas.

#### Instalación de pysnmp en Python.
Tener en cuenta que pysnmp no es un agente SNMP en sí, sino una librería de Python que permite construir agentes, gestores, traps, y más usando SNMP.
Si está snmpd activado en Ubuntu, usar pysnmp desde otro equipo (o el mismo en otra terminal) 
**NOTA: Directorios indicados son sólo a modo de ejemplo**

In [13]:
import pysnmp
print(pysnmp.__version__)

5.0.0


In [1]:
# Prueba de SNMP usando python

# getCmd: función que permite realizar una consulta SNMP tipo GET (obtener datos de un agente SNMP remoto).
# SnmpEngine: clase que representa el motor SNMP (permite cualquier operación SNMP con pysnmp).
# getCmd es una función.
# SnmpEngine es una clase.

from pysnmp.hlapi import getCmd, SnmpEngine
print(getCmd, SnmpEngine)

<function getCmd at 0x72834e904c10> <class 'pysnmp.entity.engine.SnmpEngine'>


In [2]:
# verifico la ruta del ejecutable de Python que se está utilizando en el entorno actual. Deben coincidir.
# Si el sistema tiene otro python por defecto, pero el script está usando uno diferente (el del entorno virtual), se van a producir errores.
!which python
import sys
print(sys.executable)

/home/javier/Documents/GitRepo/UCA/PdI/Notebooks/proyecto_snmp/snmp_env/bin/python
/home/javier/Documents/GitRepo/UCA/PdI/Notebooks/proyecto_snmp/snmp_env/bin/python


In [3]:
# verifico que el paquete pysnmp esté instalado correctamente en entorno de Python y que puedo importar el módulo hlapi (High-Level API) sin errores.
import pysnmp.hlapi
print("pysnmp funciona")

pysnmp funciona


pysnmp.hlapi (High-Level API) forma parte de la librería PySNMP y proporciona una interfaz de alto nivel para usar SNMP en Python.

| Operación     | Función `hlapi`                  | ¿Qué hace?                                         |
| ------------- | -------------------------------- | -------------------------------------------------- |
| GET           | `getCmd`                         | Consulta un OID específico                         |
| GETNEXT       | `nextCmd`                        | Consulta el siguiente OID (como `snmpwalk`)        |
| SET           | `setCmd`                         | Cambia el valor de un OID                          |
| TRAP (envío)  | `sendNotification`               | Envía traps o informs SNMP                         |
| Engine config | `SnmpEngine`, `CommunityData`... | Define cómo se comunica y autentica la sesión SNMP |


Script Python básico (GET):

In [16]:
from pysnmp.entity.engine import SnmpEngine
from pysnmp.hlapi import getCmd, CommunityData, UdpTransportTarget, ContextData, ObjectType, ObjectIdentity

iterator = getCmd(
    SnmpEngine(),
    CommunityData('public'),
    UdpTransportTarget(('127.0.0.1', 161)),
    ContextData(),
    ObjectType(ObjectIdentity('1.3.6.1.2.1.1.1.0'))  # sysDescr
)

errorIndication, errorStatus, errorIndex, varBinds = next(iterator)

if errorIndication:
    print(f"Error: {errorIndication}")
elif errorStatus:
    print(f"Error: {errorStatus.prettyPrint()}")
else:
    for varBind in varBinds:
        print(f"{varBind[0]} = {varBind[1]}")


1.3.6.1.2.1.1.1.0 = 


Ejecutamos el mismo comando usando snmpget bajo en linux

In [17]:
!snmpget -v2c -c public localhost:1161 1.3.6.1.2.1.1.1.0

Bad operator (INTEGER): At line 73 in /usr/share/snmp/mibs/ietf/SNMPv2-PDU
Bad operator (FROM): At line 9 in /usr/share/snmp/mibs/MY-SENSOR-MIB.txt
Timeout: No Response from localhost:1161.


#### Creación de una MIB

##### Breve descripción de teórica de la MIB
Una MIB es un archivo de texto que define objetos SNMP estructurados jerárquicamente. Se puede crear un módulo OIDs propios, por ejemplo para simular sensores, dispositivos, etc.  
Una MIB (Management Information Base) es un archivo que define objetos que pueden ser gestionados en un dispositivo mediante el protocolo SNMP (Simple Network Management Protocol). Las MIBs permiten a las herramientas de monitoreo (como Zabbix, Nagios, PRTG, etc.) entender qué datos puede consultar o controlar en un dispositivo de red (como un router, switch, servidor, etc.).   
Es un archivo de texto escrito en un lenguaje formal basado en ASN.1 (Abstract Syntax Notation One). 
Este lenguaje define:
- Objetos gestionables
- Nombres simbólicos
- Tipos de datos
- Ubicaciones dentro de una jerarquía de objetos (OID)

**Encabezado del módulo**   
Define el nombre del módulo y sus dependencias.
```json
MY-MIB DEFINITIONS ::= BEGIN

IMPORTS
    MODULE-IDENTITY, OBJECT-TYPE, Integer32
        FROM SNMPv2-SMI
    TEXTUAL-CONVENTION
        FROM SNMPv2-TC;
```
**Identidad del módulo (MODULE-IDENTITY)**
Define quién creó la MIB y otra información administrativa.
```json
myMIB MODULE-IDENTITY
    LAST-UPDATED "202405190000Z"
    ORGANIZATION "Mi Empresa"
    CONTACT-INFO "soporte@miempresa.com"
    DESCRIPTION "MIB personalizada para sensores de invernadero."
    ::= { enterprises 99999 }
```
99999 es un OID asignado a la empresa por IANA, o un número privado en pruebas.

**Definiciones de objetos (OBJECT-TYPE)**
Cada objeto representa una métrica o configuración accesible por SNMP.
```json
temperaturaSensor OBJECT-TYPE
    SYNTAX      Integer32
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION "Temperatura medida por el sensor en grados Celsius."
    ::= { myMIB 1 }

```
- SYNTAX: tipo de dato (Integer32, OCTET STRING, etc.)
- MAX-ACCESS: read-only, read-write, not-accessible
- STATUS: current, deprecated, obsolete
- ::= { myMIB 1 }: asigna un OID relativo al módulo

**Jerarquía de OIDs (Object Identifiers)**
Los OIDs son rutas jerárquicas, como direcciones, que identifican objetos.
```text
1.3.6.1.4.1.99999.1
│ │ │ │ │ └──── Empresa privada (enterprise ID asignado por IANA)
│ │ │ │ └────── Internet privado
│ │ │ └──────── MIB-II
│ │ └────────── ISO Identificado por SNMP
│ └──────────── Organización internacional
└────────────── ISO
```
  
**Estructura mínima de una MIBm y ejemplos**

```json
MY-SENSOR-MIB DEFINITIONS ::= BEGIN

IMPORTS
    MODULE-IDENTITY, OBJECT-TYPE, Integer32
        FROM SNMPv2-SMI;

mySensorMIB MODULE-IDENTITY
    LAST-UPDATED "202405190000Z"
    ORGANIZATION "Mi Empresa"
    DESCRIPTION "MIB personalizada para sensores de temperatura."
    ::= { enterprises 99999 }

temperatura OBJECT-TYPE
    SYNTAX      Integer32
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION "Temperatura en grados Celsius."
    ::= { mySensorMIB 1 }

humedad OBJECT-TYPE
    SYNTAX      Integer32
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION "Humedad relativa en porcentaje."
    ::= { mySensorMIB 2 }

END
```

```json
MY-DEMO-MIB DEFINITIONS ::= BEGIN

IMPORTS
    MODULE-IDENTITY, OBJECT-TYPE, enterprises FROM SNMPv2-SMI;

myDemoMIB MODULE-IDENTITY
    LAST-UPDATED "202505120000Z"
    ORGANIZATION "Nombre o Institución"
    CONTACT-INFO "un.email@dominio.com"
    DESCRIPTION "MIB de ejemplo."
    ::= { enterprises 99999 }

-- Definimos un nodo llamado demoText

demoText OBJECT-TYPE
    SYNTAX      OCTET STRING
    MAX-ACCESS  read-write
    STATUS      current
    DESCRIPTION "Texto de prueba"
    ::= { myDemoMIB 1 }

END
```

##### Creo una MIB para un sensor de temperatura y humedad, la registro con snmpd, y la consulto desde un cliente Python

Ubicación (verificar): /usr/share/snmp/mibs/ietf/MY-SENSOR-MIB.txt

```json
MY-SENSOR-MIB DEFINITIONS ::= BEGIN

IMPORTS
    MODULE-IDENTITY, OBJECT-TYPE, Integer32
        FROM SNMPv2-SMI
    DisplayString
        FROM SNMPv2-TC;

mySensorMIB MODULE-IDENTITY
    LAST-UPDATED "202405210000Z"
    ORGANIZATION "TuNombre o Empresa"
    CONTACT-INFO "Contacto"
    DESCRIPTION "MIB para sensores de temperatura y humedad"
    ::= { enterprises 99999 }

-- Definimos los OID base (usar un número OID único para evitar conflictos)

mySensorObjects OBJECT IDENTIFIER ::= { mySensorMIB 1 }

temperature OBJECT-TYPE
    SYNTAX      Integer32
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION "Temperatura en grados Celsius"
    ::= { mySensorObjects 1 }

humidity OBJECT-TYPE
    SYNTAX      Integer32
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION "Humedad relativa en porcentaje"
    ::= { mySensorObjects 2 }

END
```

- Script para simular valores   
  -Creamos un script en bash o Python que imprima valores numéricos.
  - Ubicarlo en: /usr/local/bin/sensor_data.sh:

Configurar snmpd.   
Editar /etc/snmp/snmpd.conf y agregar:

Esto registra dos líneas como salida del comando, cada una accesible vía OID .1.3.6.1.4.1.99999.1.1.1.1 (temperatura) y .1.3.6.1.4.1.99999.1.1.1.2 (humedad), bajo el mecanismo NET-SNMP-EXTEND-MIB.

Reiniciar el agente SNMP

Probar desde consola

##### Realizar la consulta desde python con pysnmp

In [1]:
from pysnmp.hlapi import *

def get_oid(host, community, oid):
    iterator = getCmd(SnmpEngine(),
                      CommunityData(community),
                      UdpTransportTarget((host, 161)),
                      ContextData(),
                      ObjectType(ObjectIdentity(oid)))
    
    errorIndication, errorStatus, errorIndex, varBinds = next(iterator)
    
    if errorIndication:
        print("Error:", errorIndication)
    elif errorStatus:
        print("SNMP Error:", errorStatus.prettyPrint())
    else:
        for varBind in varBinds:
            print(f"{varBind[0]} = {varBind[1]}")

# OIDs
TEMP_OID = '1.3.6.1.4.1.99999.1.1.1.1'
HUM_OID  = '1.3.6.1.4.1.99999.1.1.1.2'

get_oid('localhost', 'public', TEMP_OID)
get_oid('localhost', 'public', HUM_OID)


1.3.6.1.4.1.99999.1.1.1.1 = 
1.3.6.1.4.1.99999.1.1.1.2 = 


### Notas:
- Es posible que falten MIBs que no son cargadas automaticamente.
- Verificar destino: /usr/share/snmp/mibs/ietf
- El destino puede variar, apuntar a dónde estén todas las MIBs
- Descargarlas con los siguientes comandos:

#### MY-SENSOR-MIB

```json
MY-SENSOR-MIB DEFINITIONS ::= BEGIN

IMPORTS
    MODULE-IDENTITY, OBJECT-TYPE, Integer32, enterprises
        FROM SNMPv2-SMI;
    DisplayString
        FROM SNMPv2-TC;

mySensorMIB MODULE-IDENTITY
    LAST-UPDATED "202406090000Z"
    ORGANIZATION "Mi Empresa"
    CONTACT-INFO
        "Email: soporte@miempresa.com"
    DESCRIPTION
        "MIB para sensores de temperatura y humedad de ejemplo"
    ::= { enterprises 99999 }

mySensorObjects OBJECT IDENTIFIER ::= { mySensorMIB 1 }

temperatura OBJECT-TYPE
    SYNTAX      Integer32
    UNITS       "degrees Celsius"
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION
        "Temperatura medida por el sensor."
    ::= { mySensorObjects 1 }

humedad OBJECT-TYPE
    SYNTAX      Integer32
    UNITS       "percent"
    MAX-ACCESS  read-only
    STATUS      current
    DESCRIPTION
        "Humedad relativa medida por el sensor."
    ::= { mySensorObjects 2 }

END
```

### Aacceder por SNMP a un comando extendido, configurado como:

#### extend temperatura /bin/echo 25

- Esto le dice a Net-SNMP que cada vez que alguien consulte el valor de "temperatura", ejecute /bin/echo 25 y devuelva eso como salida.
- Este comando se vuelve visible a través de la MIB NET-SNMP-EXTEND-MIB, concretamente con objetos como:
  - nsExtendOutput1Line."temperatura" → devuelve una línea de salida del comando
  - nsExtendOutputFull."temperatura" → devuelve toda la salida del comando

#### snmpget -v2c -c public localhost 1.3.6.1.4.1.8072.1.3.2.1.2.temperatura

- Da error porque SNMP no entiende "temperatura" directamente dentro de una OID numérica. 
- En SNMP, los índices tipo OCTET STRING (como "temperatura") se codifican así:
- El primer número indica cuántos caracteres tiene la palabra (en este caso 11)
- Luego se listan los códigos ASCII de cada carácter:

Letra	ASCII
t	116
e	101
m	109
p	112
...	...
e	101

Entonces:

"temperatura" = 11.116.101.109.112.101.114.97.116.117.114.101

- La OID final completa sería:

1.3.6.1.4.1.8072.1.3.2.1.2.11.116.101.109.112.101.114.97.116.117.114.101

- ¿Qué significa esta OID?

1.3.6.1.4.1.8072.1.3.2.1.2 → nsExtendOutput1Line

Esta parte es fija, y representa el objeto que da una línea de salida del comando extendido.

11.116.101.109.112.101.114.97.116.117.114.101 → índice "temperatura"

- Esto representa el índice "temperatura" en el formato SNMP válido.
- Cómo lo consulto sin usar ese formato

**snmpget -v2c -c public -m +NET-SNMP-EXTEND-MIB localhost 'NET-SNMP-EXTEND-MIB::nsExtendOutput1Line."temperatura"'**

- Resultado: NET-SNMP-EXTEND-MIB::nsExtendOutput1Line."temperature" = STRING: 25

snmp_sensor_sim.sh

```c
#!/bin/bash
read oid
case "$oid" in
  1) echo "INTEGER: 25" ;; # temperatura 25°C
  2) echo "INTEGER: 60" ;; # humedad 60%
  *) echo "NONE" ;;
esac
```

snmpd.conf

rocommunity public localhost
sysLocation "Test Server"
sysContact admin@example.com
view   systemonly  included   .1.3.6.1.4.1.99999
pass .1.3.6.1.4.1.99999.1.1 /usr/local/bin/snmp_sensor_sim.sh
extend temperatura /bin/echo 25
extend humedad /bin/echo 60

snmp.conf

mibs +ALL
mibdirs +/usr/share/snmp/mibs:/usr/share/snmp/mibs/ietf:/home/javier/Documents/GitRepo/UCA/PdI/Notebooks/proyecto_snmp
mibs +MY-SENSOR-MIB

#### Descartar los siguientes errores o cargar manualmente esas MIBs